# Task 2: Data Preprocessing for Regression

## Section 1: Data Cleaning

### 1a. Load Data & Initial Inspection

- Load the dataset and check its shape and column types before any cleaning
- This establishes the baseline row count for later comparison

In [42]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/final_internship_data.csv")
df.shape

(500000, 26)

In [43]:
df.dtypes

,0
User ID,object
User Name,object
Driver Name,object
Car Condition,object
Weather,object
Traffic Condition,object
key,object
fare_amount,float64
pickup_datetime,object
pickup_longitude,float64


### 1b. Handling Missing Values

- Check missingness percentage per column
- Decides whether imputation is needed at all

In [44]:
df.isnull().mean().sort_values(ascending=False) * 100

,0
dropoff_latitude,0.001
bearing,0.001
jfk_dist,0.001
ewr_dist,0.001
lga_dist,0.001
sol_dist,0.001
distance,0.001
nyc_dist,0.001
dropoff_longitude,0.001
User ID,0.000


- Missingness is negligible (~0.001%) across all columns
- Decision: no imputation needed, safe to ignore

### 1c. Handling Duplicates

- Check exact duplicate rows and duplicate trip IDs (`key`)

In [45]:
print(df.duplicated().sum())
print(df['key'].duplicated().sum())

0
0


- Zero duplicates
- Decision: no rows dropped at this step

### 1d. Detecting Invalid / Impossible Values

- Check for physically impossible values: negative/zero fares, zero passengers, unrealistic distances, out-of-range coordinates
- These are data errors, not true missing values or true outliers

In [46]:
print(df['fare_amount'].describe())
print(df['distance'].describe())
print(df['jfk_dist'].describe())
print(df['ewr_dist'].describe())
print(df['lga_dist'].describe())
print(df['sol_dist'].describe())
print(df['nyc_dist'].describe())
print(df['passenger_count'].describe())
print(df['bearing'].describe())
print(df['year'].describe())
print(df[['pickup_longitude','pickup_latitude','dropoff_longitude','dropoff_latitude']].describe())

count    500000.000000
mean         11.358361
std           9.916617
min         -44.900000
25%           6.000000
50%           8.500000
75%          12.500000
max         500.000000
Name: fare_amount, dtype: float64
count    499995.000000
mean         19.468775
std         367.299601
min           0.000000
25%           1.214550
50%           2.116970
75%           3.890070
max       12399.956433
Name: distance, dtype: float64
count    499995.000000
mean        385.279367
std        2419.087483
min           1.017646
25%          41.341514
50%          42.523163
75%          43.785649
max       30133.067880
Name: jfk_dist, dtype: float64
count    499995.000000
mean        380.503657
std        2428.804740
min           1.460945
25%          32.173712
50%          34.787507
75%          38.304502
max       30167.595967
Name: ewr_dist, dtype: float64
count    499995.000000
mean        363.843772
std        2425.075903
min           0.382119
25%          17.100762
50%          19.591554

In [47]:
print((df['distance'] < 0).sum())
print((df['fare_amount'] <= 0).sum())
print((df['passenger_count'] == 0).sum())

for col in ['jfk_dist','ewr_dist','lga_dist','sol_dist','nyc_dist']:
    print(col, (df[col] < 0).sum())

for col in ['pickup_longitude','pickup_latitude','dropoff_longitude','dropoff_latitude']:
    print(col, ((df[col] < -np.pi) | (df[col] > np.pi)).sum())

0
35
1796
jfk_dist 0
ewr_dist 0
lga_dist 0
sol_dist 0
nyc_dist 0
pickup_longitude 8
pickup_latitude 6
dropoff_longitude 6
dropoff_latitude 5


### 1e. Removing Invalid Rows

- Combine all invalid conditions: distances over 200km, fare <= 0, zero passengers, out-of-range coordinates
- ~2.4% of rows (12,158) are affected, small enough to drop rather than impute
- Decision: drop, since these represent data errors, not legitimate missing/outlier data

In [48]:
bad_jfk_dist = df['jfk_dist'] > 200
bad_ewr_dist = df['ewr_dist'] > 200
bad_lga_dist = df['lga_dist'] > 200
bad_sol_dist = df['sol_dist'] > 200
bad_nyc_dist = df['nyc_dist'] > 200
bad_distance = df['distance'] > 200
bad_fare = df['fare_amount'] <= 0
bad_passengers = df['passenger_count'] == 0
bad_pickup_long = (df['pickup_longitude'] < -np.pi) | (df['pickup_longitude'] > np.pi)
bad_pickup_lat = (df['pickup_latitude'] < -np.pi) | (df['pickup_latitude'] > np.pi)

all_invalid = (
    bad_jfk_dist | bad_ewr_dist | bad_lga_dist | bad_sol_dist | bad_nyc_dist |
    bad_distance | bad_fare | bad_passengers | bad_pickup_long | bad_pickup_lat
)
print("total invalid rows:", all_invalid.sum())

total invalid rows: 12273


In [50]:
df = df[~all_invalid].reset_index(drop=True)
df.shape

/tmp/ipykernel_5542/549995519.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~all_invalid].reset_index(drop=True)


(475772, 26)

### 1f. Fixing Unit Errors (Radians -> Degrees)

- Pickup/dropoff latitude and longitude are stored in radians, convert to degrees
- Bearing is stored as a signed radian angle (-pi to pi), convert to degrees in the standard 0-360 degree range

In [51]:
coord_cols = ['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude']
for col in coord_cols:
    df[col] = np.degrees(df[col])

df['bearing'] = np.degrees(df['bearing'])
df['bearing'] = (df['bearing'] + 360) % 360

df[coord_cols + ['bearing']].describe()

,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,bearing
count,475772.000000,475772.000000,475772.000000,475772.000000,475772.000000
mean,-73.975383,40.750807,-73.974433,40.751205,203.828808
std,0.037700,0.029538,0.036801,0.032617,105.633324
min,-75.437825,39.603178,-75.448579,39.603429,0.000000
25%,-73.992272,40.736527,-73.991577,40.735599,134.101057
50%,-73.982083,40.753352,-73.980591,40.753866,186.252858
75%,-73.968348,40.767463,-73.965309,40.768407,315.015656
max,-73.085745,41.650000,-72.196091,41.505343,359.998528


### 1g. Outlier Detection (IQR) on `fare_amount` and `distance`

- Now that impossible values are removed, check for statistical outliers using the IQR rule
- This is separate from 1e: those were data errors, these are extreme-but-possibly-real values

In [52]:
for col in ['fare_amount', 'distance']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(col, "lower:", lower, "upper:", upper, "outliers:", n_outliers)

fare_amount lower: -3.75 upper: 22.25 outliers: 41124
distance lower: -2.750755577384089 upper: 7.935467301288028 outliers: 39525


- `fare_amount`: IQR range flags 42,219 rows (~8.9%) as outliers
- `distance`: IQR range flags 40,553 rows (~8.5%) as outliers
- Both columns are heavily right-skewed (fare skew = 4.87)
- These are not data errors (already removed in 1e) , they are the real, rare-but-valid data

### 1h. Outlier Handling Decision

- Decision: **keep all rows**, no deletion : these are real trips, not data errors, and removing ~9% of the data would bias the model against learning the expensive/long-distance cases
- Instead of deleting, outliers will be **capped (winsorized)**, not removed
- Capping will use custom percentile clipping (1st/99th percentile) for explicit control per column
- This must be fit on training data only and applied to test data with the same thresholds, so it belongs in Section 3, not here , computing percentiles on the full dataset before splitting would leak test information into training


## Section 2: Feature Engineering & Selection

### 2a. Dropping Redundant Datetime Column

- `pickup_datetime` is redundant since hour, day, month, weekday, and year are already extracted as separate columns

In [53]:
df = df.drop(columns=['pickup_datetime'])
df.shape

(475772, 25)

### 2b. Verifying `day` vs `weekday` Are Distinct

- Check whether these two columns are duplicates of each other before deciding to drop either

In [54]:
print(df['day'].unique())
print(df['weekday'].unique())

[15  5 18 21  9  6 20  4  3  2  8 19  7 12 10 28 11 24 29 22 31  1 14 23
 16 17 27 25 26 13 30]
[0 1 3 5 2 6 4]


- `day` = day of month (1-31), `weekday` = day of week (0-6)
- Genuinely different fields, not duplicates

### 2c. Target Skewness Check (`fare_amount`)

- Check the shape of the target distribution to decide if a transform is needed for linear models

In [55]:
print(df['fare_amount'].skew())
df['fare_amount'].describe()

4.891804308234465


,fare_amount
count,475772.000000
mean,11.350486
std,9.851443
min,0.010000
25%,6.000000
50%,8.500000
75%,12.500000
max,500.000000


- Skew = 4.87, heavily right-skewed
- Decision: apply `log1p` for linear models only, at modeling time (not here)


### 2d. Correlation Among Distance Features

- Check for redundancy between the five distance columns before feature selection

In [56]:
dist_cols = ['jfk_dist', 'ewr_dist', 'lga_dist', 'sol_dist', 'nyc_dist', 'distance']
df[dist_cols].corr()

,jfk_dist,ewr_dist,lga_dist,sol_dist,nyc_dist,distance
jfk_dist,1.000000,-0.245846,0.104356,-0.041335,-0.069818,-0.245122
ewr_dist,-0.245846,1.000000,-0.245354,0.958856,0.944350,0.497975
lga_dist,0.104356,-0.245354,1.000000,-0.254570,-0.153230,0.262393
sol_dist,-0.041335,0.958856,-0.254570,1.000000,0.985661,0.439786
nyc_dist,-0.069818,0.944350,-0.153230,0.985661,1.000000,0.492284
distance,-0.245122,0.497975,0.262393,0.439786,0.492284,1.000000


- `ewr_dist`, `sol_dist`, `nyc_dist` are highly correlated (0.95-0.99), near redundant
- Decision: keep `jfk_dist` and `lga_dist` (distinct info), keep `nyc_dist` only (most general), drop `ewr_dist` and `sol_dist`

In [57]:
df = df.drop(columns=['ewr_dist', 'sol_dist'])
df.shape

(475772, 23)

### 2e. Correlation With Target (Feature Relevance)

- Check correlation of all numeric columns with `fare_amount` to spot weak predictors

In [58]:
df.corr(numeric_only=True)['fare_amount'].sort_values(ascending=False)

,fare_amount
fare_amount,1.000000
distance,0.755025
nyc_dist,0.395752
pickup_longitude,0.379818
dropoff_longitude,0.302749
lga_dist,0.122716
year,0.115866
month,0.024389
passenger_count,0.013637
weekday,0.003389


### 2f. Categorical Features vs Fare

- Check average fare across categories for `Car Condition`, `Weather`, `Traffic Condition`

In [59]:
print(df.groupby('Car Condition')['fare_amount'].mean())
print(df.groupby('Weather')['fare_amount'].mean())
print(df.groupby('Traffic Condition')['fare_amount'].mean())

Car Condition
Bad          11.312566
Excellent    11.346975
Good         11.330699
Very Good    11.411542
Name: fare_amount, dtype: float64
Weather
cloudy    11.377441
rainy     11.344122
stormy    11.340283
sunny     11.351724
windy     11.338797
Name: fare_amount, dtype: float64
Traffic Condition
Congested Traffic    11.380532
Dense Traffic        11.359822
Flow Traffic         11.311021
Name: fare_amount, dtype: float64


- No meaningful difference in average fare across categories for any of the three, no real predictive signal
- Decision: drop all three rather than encoding them

In [60]:
df = df.drop(columns=['Car Condition', 'Weather', 'Traffic Condition'])
df.shape

(475772, 20)

### 2g. Dropping Identifiers & Low-Signal Features

- Identifier columns (`User ID`, `User Name`, `Driver Name`, `key`) have no predictive value
- `passenger_count` has negligible correlation with fare (0.013)

In [61]:
df = df.drop(columns=['User ID', 'User Name', 'Driver Name', 'key', 'passenger_count'])
df.shape

(475772, 15)

### 2h. Cyclical Encoding

- `hour`, `weekday`, `month`, and `bearing` are circular variables, e.g. hour 23 and hour 0 are close in reality but raw integers treat them as far apart
- Apply sin/cos encoding, then drop the raw columns

In [62]:
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

df['weekday_sin'] = np.sin(2 * np.pi * df['weekday'] / 7)
df['weekday_cos'] = np.cos(2 * np.pi * df['weekday'] / 7)

df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

df['bearing_sin'] = np.sin(2 * np.pi * df['bearing'] / 360)
df['bearing_cos'] = np.cos(2 * np.pi * df['bearing'] / 360)

df = df.drop(columns=['hour', 'weekday', 'month', 'bearing'])
df.shape

(475772, 19)

## Section 3 : Train/Test Split

3a. Train/Test Split

- Split into train/test before any capping, fitting, or transforming very later step must only ever learn from X_train

In [63]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['fare_amount'])
y = df['fare_amount']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

(380617, 18) (95155, 18)


4a. Custom Percentile Capper (Outlier Handling from 1h)

Implements the capping decision from Section 1h: clip fare_amount-related outliers in distance at the 1st/99th percentile

In [64]:
from sklearn.base import BaseEstimator, TransformerMixin

class PercentileCapper(BaseEstimator, TransformerMixin):
    def __init__(self, columns, lower_pct=0.01, upper_pct=0.99):
        self.columns = columns
        self.lower_pct = lower_pct
        self.upper_pct = upper_pct

    def fit(self, X, y=None):
        self.bounds_ = {}
        for col in self.columns:
            lower = X[col].quantile(self.lower_pct)
            upper = X[col].quantile(self.upper_pct)
            self.bounds_[col] = (lower, upper)
        return self

    def transform(self, X):
        X = X.copy()
        for col, (lower, upper) in self.bounds_.items():
            X[col] = X[col].clip(lower, upper)
        return X

In [65]:
capper = PercentileCapper(columns=['distance'])

capper.fit(X_train)

X_train_capped = capper.transform(X_train)
X_test_capped = capper.transform(X_test)

print(X_train['distance'].describe())
print(X_train_capped['distance'].describe())

count    380617.000000
mean          3.355228
std           4.099953
min           0.000000
25%           1.255232
50%           2.154017
75%           3.926057
max         173.852082
Name: distance, dtype: float64
count    380617.000000
mean          3.309070
std           3.512240
min           0.000000
25%           1.255232
50%           2.154017
75%           3.926057
max          20.361153
Name: distance, dtype: float64


4c. Log1p Transform on Target (fare_amount)

Applying log1p to compress the right-skew (skew = 4.87) we found in Section 2c
This is a fixed, row-independent transform (not fitted), so it can safely be applied to y_train and y_test directly — no leakage risk, unlike the capper

In [66]:
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

print(y_train.skew())
print(y_train_log.skew())

5.009639242059099
0.9806670198151858


4d. Wrapping the Capper in a Pipeline

In [67]:
from sklearn.pipeline import Pipeline

preprocessing_pipeline = Pipeline([
    ('capper', PercentileCapper(columns=['distance']))
])

preprocessing_pipeline.fit(X_train)

X_train_final = preprocessing_pipeline.transform(X_train)
X_test_final = preprocessing_pipeline.transform(X_test)

X_train_final.shape, X_test_final.shape

((380617, 18), (95155, 18))

## Section 5 : Training Decision Tree Model

 5a. Baseline Model: Decision Tree Regressor

- max_depth — how deep the tree can grow
- min_samples_split — minimum samples required to split a node (higher = less overfitting)
- min_samples_leaf — minimum samples required in a leaf (higher = smoother, less noisy splits)
- max_features — how many features are considered at each split (adds randomness, reduces variance)

- Using RandomizedSearchCV

In [28]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import RandomizedSearchCV

dt_param_grid = {
    'max_depth': [4, 6, 8, 10, 12, 15],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 5, 10, 20],
    'max_features': [None, 'sqrt', 'log2']
}

dt_search = RandomizedSearchCV(
    estimator=DecisionTreeRegressor(random_state=42),
    param_distributions=dt_param_grid,
    n_iter=20,
    cv=5,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1
)

dt_search.fit(X_train_final, y_train_log)

print(dt_search.best_params_)
print(dt_search.best_score_)

{'min_samples_split': 5, 'min_samples_leaf': 20, 'max_features': None, 'max_depth': 10}
-0.2350128102516184


## Section 5 : Training XGBoost Model

- n_estimators — number of boosting rounds
- max_depth — depth of each individual tree (kept shallower than a standalone tree, since boosting adds many of them)
- learning_rate — how much each tree corrects the previous ones
- subsample — fraction of rows used per tree (adds regularization)
- colsample_bytree — fraction of features used per tree (adds regularization)

- Same RandomizedSearchCV approach as the Decision Tree

In [29]:
from xgboost import XGBRegressor

xgb_param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 4, 5, 6, 8],
    'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

xgb_search = RandomizedSearchCV(
    estimator=XGBRegressor(random_state=42, n_jobs=-1),
    param_distributions=xgb_param_grid,
    n_iter=20,
    cv=5,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=1
)

xgb_search.fit(X_train_final, y_train_log)

print(xgb_search.best_params_)
print(xgb_search.best_score_)

{'subsample': 1.0, 'n_estimators': 200, 'max_depth': 8, 'learning_rate': 0.1, 'colsample_bytree': 0.8}
-0.20842844081640574


7a. Model Comparison: Decision Tree vs. XGBoost



In [30]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

dt_best = dt_search.best_estimator_
xgb_best = xgb_search.best_estimator_

dt_preds_log = dt_best.predict(X_test_final)
xgb_preds_log = xgb_best.predict(X_test_final)

dt_preds = np.expm1(dt_preds_log)
xgb_preds = np.expm1(xgb_preds_log)

y_test_actual = np.expm1(y_test_log)

results = pd.DataFrame({
    'Model': ['Decision Tree (baseline)', 'XGBoost (tuned)'],
    'MAE': [
        mean_absolute_error(y_test_actual, dt_preds),
        mean_absolute_error(y_test_actual, xgb_preds)
    ],
    'RMSE': [
        np.sqrt(mean_squared_error(y_test_actual, dt_preds)),
        np.sqrt(mean_squared_error(y_test_actual, xgb_preds))
    ],
    'R2': [
        r2_score(y_test_actual, dt_preds),
        r2_score(y_test_actual, xgb_preds)
    ]
})

results

,Model,MAE,RMSE,R2
0,Decision Tree (baseline),1.906349,4.571158,0.788603
1,XGBoost (tuned),1.561713,4.283424,0.814379


# Pipeline

In [31]:
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from xgboost import XGBRegressor
import joblib

final_model = XGBRegressor(**xgb_search.best_params_, random_state=42, n_jobs=-1)

target_transformed_model = TransformedTargetRegressor(
    regressor=final_model,
    func=np.log1p,
    inverse_func=np.expm1
)

final_pipeline = Pipeline([
    ('capper', PercentileCapper(columns=['distance'])),
    ('model', target_transformed_model)
])

final_pipeline.fit(X_train, y_train)

joblib.dump(final_pipeline, 'fare_prediction_pipeline.pkl')

print("Pipeline saved.")

Pipeline saved.


In [32]:
loaded_pipeline = joblib.load('fare_prediction_pipeline.pkl')

pipeline_preds = loaded_pipeline.predict(X_test)

pipeline_mae = mean_absolute_error(y_test, pipeline_preds)
pipeline_rmse = np.sqrt(mean_squared_error(y_test, pipeline_preds))
pipeline_r2 = r2_score(y_test, pipeline_preds)

print("MAE:", pipeline_mae)
print("RMSE:", pipeline_rmse)
print("R2:", pipeline_r2)

MAE: 1.5617129803109495
RMSE: 4.283424319870584
R2: 0.8143785709685705


# Final Notes

 Note on Encoding & Scaling

- Encoding: all categorical columns (Car Condition, Weather, Traffic Condition) were dropped in 2f for showing no meaningful relationship with fare_amount. What remains in X is 100% numeric , there is nothing left to encode
- Scaling: both baseline (Decision Tree) and tuned (XGBoost) models are tree-based. Tree splits are based on thresholds, not weighted sums, so they are unaffected by feature scale , scaling would add computation with zero benefit


#What I Would Try Next
- Try Gradient Boosting or LightGBM as a third comparison point , since XGBoost already outperformed the Decision Tree, adding one more boosting-family model would clarify whether XGBoost's specific implementation mattered, or whether any modern boosting method would land in the same range.
- Revisit the capping thresholds ,  1st/99th percentile was a reasonable default, but testing 2nd/98th or 5th/95th on distance could reveal whether the cap is too aggressive or too loose for this specific dataset's tail shape.

- Try target/frequency encoding on a re-included categorical feature , Car Condition, Weather, and Traffic Condition were dropped for showing no average fare difference (2f), but a groupby on the mean can hide interaction effects (e.g. bad weather might only matter combined with rush hour). Worth revisiting with an interaction feature instead of dropping outright.